# Agentic Systems vLLM release attestation

This Colab notebook certifies the exact release-candidate wheel against all four frameworks. Upload the wheel produced by GitHub Actions, set the exact commit SHA and choose a vLLM-compatible model. The resulting JSON is valid for 24 hours and contains no credentials.

In [ ]:
from pathlib import Path
from google.colab import files

REPOSITORY_URL = 'https://github.com/JacoboGGLeon/agentic_systems.git'
COMMIT_SHA = ''  # Paste the full candidate commit SHA.
VLLM_MODEL = 'Qwen/Qwen3-4B-Instruct-2507'  # Replace if the GPU requires another model.
uploaded = files.upload()
wheel_names = [name for name in uploaded if name.endswith('.whl')]
assert len(wheel_names) == 1, 'Upload exactly one release-candidate wheel.'
assert len(COMMIT_SHA) == 40, 'COMMIT_SHA must be the full 40-character SHA.'
WHEEL_PATH = Path('/content') / wheel_names[0]


In [ ]:
import subprocess

subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, '/content/agentic-systems'], check=True)
subprocess.run(['git', 'checkout', COMMIT_SHA], cwd='/content/agentic-systems', check=True)
subprocess.run([
    'python', '-m', 'pip', 'install', str(WHEEL_PATH),
    'vllm>=0.9', 'openai>=2.45,<3', 'openai-agents>=0.18.3,<0.19',
    'langgraph>=0.2', 'strands-agents>=1.29,<2', 'mcp>=1,<2'
], check=True)


In [ ]:
import os
import time
import urllib.request

server_log = open('/content/vllm-server.log', 'w', encoding='utf-8')
server = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', VLLM_MODEL, '--host', '127.0.0.1', '--port', '8000'
], stdout=server_log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('vLLM did not become healthy; inspect /content/vllm-server.log')
os.environ.update(VLLM_BASE_URL='http://127.0.0.1:8000/v1', VLLM_API_KEY='vllm', VLLM_MODEL=VLLM_MODEL)


In [ ]:
import json
import platform

os.environ['GPU_NAME'] = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True
).strip().splitlines()[0]
os.environ['CUDA_VERSION'] = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True
).strip().splitlines()[0]
os.environ['VLLM_VERSION'] = subprocess.check_output(
    ['python', '-c', 'import importlib.metadata as m; print(m.version(\"vllm\"))'], text=True
).strip()
OUTPUT = Path('/content/vllm-attestation.json')
subprocess.run([
    'python', 'scripts/run_live_matrix.py', '--wheel', str(WHEEL_PATH),
    '--output', str(OUTPUT), '--commit', COMMIT_SHA,
    '--providers', 'vllm-runtime',
    '--frameworks', 'native', 'langgraph', 'openai-agents', 'strands'
], cwd='/content/agentic-systems', check=True)
evidence = json.loads(OUTPUT.read_text(encoding='utf-8'))
assert len(evidence['cases']) == 4 and all(case['ok'] for case in evidence['cases'])
print(json.dumps({'ok': True, 'python': platform.python_version(), 'output': str(OUTPUT)}, indent=2))
files.download(str(OUTPUT))
